In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import classification_report
import joblib
import json

In [ ]:
# 1.load data 
df = pd.read_csv("healthcare-dataset-stroke-data.csv")

# bỏ id (không có ý nghĩa)
df = df.drop("id", axis=1)

In [ ]:
# 2.xử lý MISSING 
# bmi bị thiếu -> sẽ xử lý bằng SimpleImputer

In [ ]:
# 3. ENCODE LABEL
categorical_cols = ["gender", "ever_married", "work_type",
                     "Residence_type", "smoking_status"]

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = df[col].astype(str)
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

In [ ]:
# 4. CHIA DATA
X = df.drop("stroke", axis=1)
y = df["stroke"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# 5. PREPROCESSOR
numeric_features = X.columns

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features)
])

In [ ]:
# 6. CÁC MODEL + GRID SEARCH
models = {
    "logistic": (
        LogisticRegression(),
        {
            "model__C": [0.1, 1, 10],
            "model__max_iter": [200]
        }
    ),

    "random_forest": (
        RandomForestClassifier(),
        {
            "model__n_estimators": [100, 200],
            "model__max_depth": [5, 10, None]
        }
    ),

    "svm": (
        SVC(),
        {
            "model__C": [0.1, 1, 10],
            "model__kernel": ["rbf", "linear"]
        }
    ),

    "knn": (
        KNeighborsClassifier(),
        {
            "model__n_neighbors": [3, 5, 7, 9]
        }
    )
}

best_models = {}
results = {}


In [ ]:
# 7. TRAIN + GRIDSEARCH
for name, (model, params) in models.items():

    pipe = Pipeline([
        ("preprocess", preprocessor),
        ("model", model)
    ])

    grid = GridSearchCV(pipe, params, cv=5, scoring="f1", n_jobs=-1)
    grid.fit(X_train, y_train)

    best_models[name] = grid.best_estimator_

    y_pred = grid.predict(X_test)

    results[name] = {
        "best_params": grid.best_params_,
        "report": classification_report(y_test, y_pred, output_dict=True)
    }

    print(f"\n===== {name.upper()} =====")
    print("Best params:", grid.best_params_)
    print(classification_report(y_test, y_pred))

In [ ]:
# 8. LƯU MODEL + THÔNG SỐ
joblib.dump(best_models, "best_models.pkl")

with open("model_results.json", "w") as f:
    json.dump(results, f, indent=4)

print("\nDone! Models + parameters saved.")